## A Complete Modelling Example - Passport Problem

The [Passport Index Dataset](https://github.com/ilyankou/passport-index-dataset) 
lists travel visa requirements for 199 countries, in .csv format.
Our task is to find out the minimum number of passports required to visit all countries.

In this dataset, the first column represents a passport (=from) and each remaining column represents a foreign country (=to). 
The values in each cell are as follows:
* 3 = visa-free travel
* 2 = eTA is required
* 1 = visa can be obtained on arrival
* 0 = visa is required
* -1 is for all instances where passport and destination are the same

Our task is to find out the minimum number of passports needed to visit every country without requiring a visa.
Thus, the values we are interested in are -1 and 3. Let us modify the data in the following manner -

In [ ]:
using CSV, DataFrames

In [ ]:
passportdata = CSV.read("data/passport-index-matrix.csv", DataFrame, copycols = true)

for i in 1:nrow(passportdata)
    for j in 2:ncol(passportdata)
        if passportdata[i,j] == -1 || passportdata[i,j] == 3
            passportdata[i,j] = 1
        else
            passportdata[i,j] = 0
        end
    end
end

In [ ]:
passportdata

The values in the cells now represent:
* 1 = no visa required for travel
* 0 = visa required for travel

Let us associate each passport with a decision variable $pass_{cntr}$ for each country. 
We want to minimize the sum $\sum pass_{cntr}$ over all countries.

Since we wish to visit all the countries, for every country, 
we should own atleast one passport that lets us travel to that country visa free. 
For one destination, this can be mathematically represented as $\sum_{cntr \in world} passportdata_{cntr,dest} \cdot pass_{cntr} \geq 1$.

Thus, we can represent this problem using the following model:

$$
\begin{align*}
\min && \sum_{cntr \in World} pass_{cntr} \\
\text{s.t.} && \sum_{cntr \in World} passportdata_{cntr,dest} \cdot pass_{cntr} \geq 1 && \forall dest \in World \\
&& pass_{cntr} \in \{0,1\} && \forall cntr \in World
\end{align*}
$$

We'll now solve the problem using JuMP.

In [ ]:
using JuMP, HiGHS

# Finding number of countries
n = ncol(passportdata) - 1 # Subtract 1 for column representing country of passport

model = Model(HiGHS.Optimizer)
@variable(model, pass[1:n], Bin)
@constraint(model, [j = 2:n], sum(passportdata[i,j] * pass[i] for i in 1:n) >= 1)
@objective(model, Min, sum(pass))
optimize!(model)

println("Minimum number of passports needed: ", objective_value(model))

In [ ]:
countryindex = findall(value.(pass) .== 1 )

print("Countries: ")
for i in countryindex
    print(names(passportdata)[i+1], " ")
end

In [ ]:
open("modelo_detalhado.txt", "w") do arquivo
    println(arquivo, "=== PROBLEMA DE OTIMIZAÇÃO ===")
    println(arquivo, "\nVARIÁVEIS:")
    for var in all_variables(model)
        # Verificar se existe lower bound
        lb = has_lower_bound(var) ? lower_bound(var) : "-∞"
        # Verificar se existe upper bound  
        ub = has_upper_bound(var) ? upper_bound(var) : "+∞"
        println(arquivo, "  ", var, " ∈ [", lb, ", ", ub, "]")
    end
    
    println(arquivo, "\nFUNÇÃO OBJETIVO:")
    println(arquivo, "  ", objective_sense(model), " ", objective_function(model))
    
    println(arquivo, "\nRESTRIÇÕES:")
    for (i, constraint) in enumerate(all_constraints(model, include_variable_in_set_constraints=true))
        println(arquivo, "  C", i, ": ", constraint)
    end
end

In [ ]:
# Finding number of countries
newpassportdata = passportdata[1:10, 1:11]
n = ncol(newpassportdata) - 1 # Subtract 1 for column representing country of passport

model = Model(HiGHS.Optimizer)
@variable(model, pass[1:n], Bin)
@constraint(model, [j = 2:n], sum(newpassportdata[i,j] * pass[i] for i in 1:n) >= 1)
@objective(model, Min, sum(pass))
optimize!(model)

println("Minimum number of passports needed: ", objective_value(model))

In [ ]:
countryindex = findall(value.(pass) .== 1 )

print("Countries: ")
for i in countryindex
    print(names(newpassportdata)[i+1], " ")
end

In [ ]:
newpassportdata

In [ ]:
print(model)
